In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Load dataset
training_data = pd.read_csv('./wine.csv')

# Separate target and encode strings to integers (good: 1, bad: 0)
training_y = training_data.pop('quality')
training_y.replace("good", 1, inplace=True)
training_y.replace("bad", 0, inplace=True)

training_x = training_data

# Convert Pandas DataFrames to NumPy float32 arrays for Keras
arr_train_x = training_x.to_numpy().astype("float32")
arr_train_y = training_y.to_numpy().astype("float32")

# Hold-out evaluation split (70% train, 30% test)
X_train, X_test, Y_train, Y_test = train_test_split(
    arr_train_x, arr_train_y, test_size=0.3, random_state=2, shuffle=True
)

model = keras.Sequential([
    layers.InputLayer(shape=(11,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.summary()

model.compile(
    loss='binary_crossentropy',
    optimizer='sgd',
    metrics=['accuracy']
)

# Train on the training set
model_training_history = model.fit(
    X_train, 
    Y_train, 
    epochs=50, 
    batch_size=32, 
    validation_data=(X_test, Y_test)
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

acc = model_training_history.history['accuracy']
loss = model_training_history.history['loss']

ax1.plot(acc, label='Train Accuracy')
ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(loss, label='Train Loss', color='red')
ax2.set_ylabel('Loss')
ax2.set_xlabel('Epochs')
ax2.legend()

plt.show()

# Predict probabilities on holdout test set
y_pred_probs = model.predict(X_test)
y_pred = np.round(y_pred_probs)  # Thresholding >= 0.5 -> 1, < 0.5 -> 0

# Confusion Matrix components
tn, fp, fn, tp = confusion_matrix(Y_test, y_pred).ravel()

print(f"TP: {tp} | FP: {fp} | TN: {tn} | FN: {fn}")

# Display matrix visually
ConfusionMatrixDisplay.from_predictions(Y_test, y_pred)
plt.show()